# 🏫 OptiPlua — Générateur de Données Synthétiques

---

## 📋 Description

Ce notebook génère des **données synthétiques réalistes** pour le projet **OptiPlua**, un système d'optimisation d'emplois du temps scolaires et universitaires.

Les données générées couvrent :
- 👨‍🏫 **Enseignants** — compétences, disponibilités, contraintes horaires
- 🏛️ **Salles** — capacités, types
- 📚 **Matières** — exigences hebdomadaires, besoins en laboratoire
- 🎓 **Classes** — niveaux, effectifs, type d'établissement

## 🏗️ Types d'établissements supportés

| Type | Description | Niveaux |
|------|-------------|----------|
| `Ecole_Standard` | Collège et lycée | Collège, Tronc Commun, Bac |
| `Centre_Soutien` | Cours privés | Collège → Prépa |
| `Universite` | Enseignement supérieur | Licence, Master, Ingénieur |

---

> **Auteur :** Projet OptiPlua  
> **Version :** 2.0.0  
> **Python :** 3.10+

---
## ⚙️ Cellule 1 — Imports, Configuration & Logging

In [ ]:
import pandas as pd
import random
import logging
import sys

# ============================================================
# Logging Configuration
# ============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
logger = logging.getLogger("OptiPlua.DataGenerator")

# ============================================================
# Reproducibility Seed
# ============================================================
RANDOM_SEED: int = 42
random.seed(RANDOM_SEED)
logger.info(f"Random seed fixé à {RANDOM_SEED} pour la reproductibilité.")

# ============================================================
# Global Configuration
# ============================================================
TYPES_ETABLISSEMENT: list[str] = [
    "Ecole_Standard",
    "Centre_Soutien",
    "Universite",
]

MATIERES_PAR_ETAB: dict[str, list[str]] = {
    "Ecole_Standard": [
        "MATH", "PHYSIQUE", "CHIMIE", "ARABE",
        "FRANCAIS", "ANGLAIS", "SVT", "PHILOSOPHIE", "INFORMATIQUE",
    ],
    "Centre_Soutien": [
        "MATH", "PHYSIQUE", "CHIMIE", "SVT", "FRANCAIS", "ANGLAIS",
    ],
    "Universite": [
        "ALGEBRE", "ANALYSE", "MECANIQUE_QUANTIQUE",
        "DROIT_CIVIL", "ECONOMIE", "BASE_DE_DONNEES",
        "RESEAUX", "BIOLOGIE",
    ],
}

NIVEAUX_PAR_ETAB: dict[str, list[str]] = {
    "Ecole_Standard": ["College", "Tronc_Commun", "Baccalaureat"],
    "Centre_Soutien": ["College", "Tronc_Commun", "Baccalaureat", "Prepa"],
    "Universite": ["Licence_1", "Licence_2", "Master_1", "Cycle_Ingenieur"],
}

CRENEAUX_HORAIRES: list[str] = [
    "Lundi_08:00-10:00",
    "Vendredi_15:00-18:00",
    "Samedi_09:00-12:00",
    "Mercredi_14:00-16:00",
]

# Matières nécessitant potentiellement un laboratoire
MATIERES_LABO: set[str] = {
    "PHYSIQUE", "CHIMIE", "SVT", "INFORMATIQUE", "BIOLOGIE",
}

logger.info("✅ Configuration et variables globales chargées avec succès.")

---
## 👨‍🏫 Cellule 2 — Générateur d'Enseignants

In [ ]:
def generer_enseignants(nb_enregistrements: int = 150) -> pd.DataFrame:
    """
    Génère un DataFrame contenant des données synthétiques d'enseignants.

    Chaque enseignant est associé à un type d'établissement, des matières,
    des niveaux autorisés, et des contraintes horaires réalistes.

    Args:
        nb_enregistrements (int): Nombre d'enseignants à générer. Défaut : 150.

    Returns:
        pd.DataFrame: DataFrame avec les colonnes :
            ID_Enseignant, Nom_Enseignant, Type_Etablissement,
            Matieres, Niveaux_Autorises, Heures_Max_Par_Semaine,
            Heures_Max_Consecutives, Creneaux_Indisponibles, Indice_Flexibilite
    """
    logger.info(f"Génération de {nb_enregistrements} enseignants...")
    data: list[dict] = []

    # Paramètres par type d'établissement
    CONFIG_ETAB = {
        "Centre_Soutien": {
            "heures_options": [8, 12, 16],
            "max_consecutif": 3,
            "flexibilite": 0.9,
        },
        "Universite": {
            "heures_options": [10, 14, 18],
            "max_consecutif": 4,
            "flexibilite": 0.6,
        },
        "Ecole_Standard": {
            "heures_options": [20, 22, 24],
            "max_consecutif": 4,
            "flexibilite": 0.2,
        },
    }

    for i in range(1, nb_enregistrements + 1):
        type_etab = random.choice(TYPES_ETABLISSEMENT)
        cfg = CONFIG_ETAB[type_etab]

        # Sélection des matières (1 ou 2 selon probabilité)
        matieres_dispo = MATIERES_PAR_ETAB[type_etab]
        nb_matieres = 2 if (random.random() < 0.2 and len(matieres_dispo) >= 2) else 1
        matieres_assignees = random.sample(matieres_dispo, nb_matieres)

        # Sélection des niveaux (1 ou 2)
        niveaux_dispo = NIVEAUX_PAR_ETAB[type_etab]
        nb_niveaux = random.choice([1, 2]) if len(niveaux_dispo) >= 2 else 1
        niveaux_assignes = random.sample(niveaux_dispo, nb_niveaux)

        # Créneaux d'indisponibilité (30% de probabilité)
        indisponibilite = (
            [random.choice(CRENEAUX_HORAIRES)] if random.random() < 0.3 else []
        )

        data.append({
            "ID_Enseignant": f"T{i:04d}",
            "Nom_Enseignant": f"Prof_{i}",
            "Type_Etablissement": type_etab,
            "Matieres": str(matieres_assignees),
            "Niveaux_Autorises": str(niveaux_assignes),
            "Heures_Max_Par_Semaine": random.choice(cfg["heures_options"]),
            "Heures_Max_Consecutives": cfg["max_consecutif"],
            "Creneaux_Indisponibles": str(indisponibilite),
            "Indice_Flexibilite": cfg["flexibilite"],
        })

    df = pd.DataFrame(data)
    logger.info(f"✅ {len(df)} enseignants générés.")
    return df


logger.info("✅ Fonction 'generer_enseignants' définie.")

---
## 🏛️ Cellule 3 — Générateurs : Salles, Matières & Classes

In [ ]:
def generer_salles(nb_enregistrements: int = 50) -> pd.DataFrame:
    """
    Génère un DataFrame de salles avec capacités et types adaptés
    à chaque type d'établissement.

    Args:
        nb_enregistrements (int): Nombre de salles à générer. Défaut : 50.

    Returns:
        pd.DataFrame: Colonnes — ID_Salle, Type_Etablissement,
                      Capacite, Type_Salle.
    """
    logger.info(f"Génération de {nb_enregistrements} salles...")
    data: list[dict] = []

    CONFIG_SALLES = {
        "Universite": {
            "capacites": [80, 150, 200],
            "types": ["Amphitheatre", "Laboratoire"],
        },
        "Centre_Soutien": {
            "capacites": [10, 15, 20],
            "types": ["Generale"],
        },
        "Ecole_Standard": {
            "capacites": [30, 35, 40],
            "types": ["Generale", "Generale", "Labo_Informatique", "Labo_Science"],
        },
    }

    for i in range(1, nb_enregistrements + 1):
        type_etab = random.choice(TYPES_ETABLISSEMENT)
        cfg = CONFIG_SALLES[type_etab]

        data.append({
            "ID_Salle": f"R{i:03d}",
            "Type_Etablissement": type_etab,
            "Capacite": random.choice(cfg["capacites"]),
            "Type_Salle": random.choice(cfg["types"]),
        })

    df = pd.DataFrame(data)
    logger.info(f"✅ {len(df)} salles générées.")
    return df


def generer_matieres() -> pd.DataFrame:
    """
    Génère un DataFrame de toutes les matières disponibles,
    regroupées par type d'établissement.

    Chaque matière reçoit un volume horaire hebdomadaire et
    une indication si un laboratoire est requis.

    Returns:
        pd.DataFrame: Colonnes — ID_Matiere, Nom_Matiere,
                      Type_Etablissement, Heures_Hebdo_Requises, Necessite_Labo.
    """
    logger.info("Génération des matières...")
    data: list[dict] = []

    for type_etab, matieres in MATIERES_PAR_ETAB.items():
        for matiere in matieres:
            # Labo requis si la matière est expérimentale (probabilité 50%)
            necessite_labo = matiere in MATIERES_LABO and random.random() < 0.5
            heures_hebdo = random.choice([2, 4, 6])

            data.append({
                "ID_Matiere": f"SUB_{type_etab.upper()}_{matiere}",
                "Nom_Matiere": matiere,
                "Type_Etablissement": type_etab,
                "Heures_Hebdo_Requises": heures_hebdo,
                "Necessite_Labo": necessite_labo,
            })

    df = pd.DataFrame(data)
    logger.info(f"✅ {len(df)} matières générées.")
    return df


def generer_classes(nb_enregistrements: int = 100) -> pd.DataFrame:
    """
    Génère un DataFrame de classes avec leurs effectifs et niveaux,
    adaptés au type d'établissement.

    Args:
        nb_enregistrements (int): Nombre de classes à générer. Défaut : 100.

    Returns:
        pd.DataFrame: Colonnes — ID_Classe, Type_Etablissement,
                      Niveau, Nombre_Etudiants.
    """
    logger.info(f"Génération de {nb_enregistrements} classes...")
    data: list[dict] = []

    CONFIG_CLASSES = {
        "Universite": {"min": 70, "max": 180},
        "Centre_Soutien": {"min": 5, "max": 15},
        "Ecole_Standard": {"min": 25, "max": 40},
    }

    for i in range(1, nb_enregistrements + 1):
        type_etab = random.choice(TYPES_ETABLISSEMENT)
        niveau = random.choice(NIVEAUX_PAR_ETAB[type_etab])
        cfg = CONFIG_CLASSES[type_etab]

        data.append({
            "ID_Classe": f"C{i:04d}",
            "Type_Etablissement": type_etab,
            "Niveau": niveau,
            "Nombre_Etudiants": random.randint(cfg["min"], cfg["max"]),
        })

    df = pd.DataFrame(data)
    logger.info(f"✅ {len(df)} classes générées.")
    return df


logger.info("✅ Fonctions 'generer_salles', 'generer_matieres', 'generer_classes' définies.")

---
## 🚀 Cellule 4 — Exécution, Export CSV & Rapport de Validation

In [ ]:
def valider_dataframe(df: pd.DataFrame, nom: str) -> None:
    """
    Valide un DataFrame généré et affiche un rapport de qualité.

    Vérifie l'absence de valeurs nulles, la cohérence du nombre
    de lignes, et affiche les statistiques de base.

    Args:
        df (pd.DataFrame): Le DataFrame à valider.
        nom (str): Nom du jeu de données (pour les logs).

    Raises:
        ValueError: Si le DataFrame contient des valeurs nulles.
    """
    nb_nulls = df.isnull().sum().sum()
    if nb_nulls > 0:
        raise ValueError(f"[{nom}] ❌ {nb_nulls} valeur(s) nulle(s) détectée(s) !")
    logger.info(f"[{nom}] ✅ Validation OK — {len(df)} lignes, {len(df.columns)} colonnes, 0 valeur nulle.")


def exporter_csv(df: pd.DataFrame, nom_fichier: str) -> None:
    """
    Exporte un DataFrame vers un fichier CSV encodé en UTF-8.

    Args:
        df (pd.DataFrame): Le DataFrame à exporter.
        nom_fichier (str): Nom du fichier de sortie (ex: 'salles_data.csv').
    """
    df.to_csv(nom_fichier, index=False, encoding="utf-8")
    logger.info(f"💾 Exporté → {nom_fichier}")


# ============================================================
# Exécution principale
# ============================================================
logger.info("=" * 55)
logger.info("  DÉMARRAGE — Génération des données synthétiques")
logger.info("=" * 55)

df_enseignants = generer_enseignants(nb_enregistrements=200)
df_salles      = generer_salles(nb_enregistrements=80)
df_matieres    = generer_matieres()
df_classes     = generer_classes(nb_enregistrements=150)

# ============================================================
# Validation des données
# ============================================================
logger.info("-" * 55)
logger.info("  VALIDATION des données générées")
logger.info("-" * 55)

valider_dataframe(df_enseignants, "Enseignants")
valider_dataframe(df_salles,      "Salles")
valider_dataframe(df_matieres,    "Matières")
valider_dataframe(df_classes,     "Classes")

# ============================================================
# Export CSV
# ============================================================
logger.info("-" * 55)
logger.info("  EXPORT vers fichiers CSV")
logger.info("-" * 55)

exporter_csv(df_enseignants, "enseignants_data.csv")
exporter_csv(df_salles,      "salles_data.csv")
exporter_csv(df_matieres,    "matieres_data.csv")
exporter_csv(df_classes,     "classes_data.csv")

logger.info("=" * 55)
logger.info("  ✅ TERMINÉ — Tous les fichiers ont été créés.")
logger.info("=" * 55)

---
## 📊 Cellule 5 — Statistiques Récapitulatives

In [ ]:
print("\n" + "=" * 55)
print("  📊 RAPPORT — Statistiques des données générées")
print("=" * 55)

datasets = {
    "👨‍🏫 Enseignants": df_enseignants,
    "🏛️  Salles      ": df_salles,
    "📚 Matières    ": df_matieres,
    "🎓 Classes     ": df_classes,
}

for label, df in datasets.items():
    print(f"\n{label} — {len(df)} enregistrements")
    if "Type_Etablissement" in df.columns:
        dist = df["Type_Etablissement"].value_counts()
        for etab, count in dist.items():
            pct = count / len(df) * 100
            bar = "█" * int(pct / 5)
            print(f"   {etab:<20} {bar:<20} {count:>4} ({pct:.1f}%)")

print("\n" + "=" * 55)
print("  Aperçu — 5 premiers enseignants")
print("=" * 55)
df_enseignants.head()